# Transformer 架构入门

本笔记对应论文：*Attention Is All You Need*（Vaswani et al., 2017）

## 目标
- 理解 Transformer 的整体结构
- 动手实现自注意力（Self-Attention）核心计算
- 理解为什么注意力机制取代了 RNN

## 1. 为什么需要 Transformer？

在 Transformer 之前，序列建模主要依赖 **RNN（循环神经网络）**：

**RNN 的两大问题：**
1. **串行计算**：必须按顺序处理，无法并行，训练慢
2. **长程依赖问题**：序列长度增大后，早期信息在隐状态中被稀释

Transformer 用 **注意力机制** 解决了这两个问题：
- 所有位置同时计算（并行）
- 任意两个位置之间的距离都是 1（直接建立连接）

## 2. Transformer 整体架构

现代 LLM（GPT 系列）只使用**解码器（Decoder-Only）**部分。

**核心组件：**
| 组件 | 作用 |
|------|------|
| Token Embedding | 将 token id 转为向量 |
| Positional Encoding | 注入位置信息（注意力本身无位置感知）|
| Multi-Head Attention | 让每个 token 关注序列中其他 token |
| Feed-Forward Network | 对每个位置独立的两层 MLP |
| Layer Norm + Residual | 稳定训练，缓解梯度消失 |

## 3. 自注意力（Self-Attention）核心计算

### 直觉理解

对于句子 "The animal did not cross the street because it was too tired"，
模型需要理解 "it" 指的是 "animal" 而不是 "street"。
自注意力让 "it" 能够直接指向 "animal" 并赋予更高权重。

### 数学公式

83268\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V83268

- **Q（Query）**：当前 token 想查询的信息
- **K（Key）**：每个 token 提供的索引标签
- **V（Value）**：每个 token 实际携带的内容
- **$\sqrt{d_k}*：缩放因子，防止点积过大导致 softmax 梯度消失

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = softmax(scores)
    output = weights @ V
    return output, weights

np.random.seed(42)
seq_len, d_k = 3, 4
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)
print("注意力权重矩阵（每行代表一个 token 对其他 token 的关注度）：")
print(np.round(weights, 3))
print("每行之和（应为 1.0）：", weights.sum(axis=-1).round(3))

## 4. 因果掩码（Causal Mask）

在 Decoder-Only LLM 中，生成第 t 个 token 时只能看到位置 ≤ t 的 token，
不能提前看到未来——这通过**下三角掩码**实现：

In [ ]:
causal_mask = np.tril(np.ones((seq_len, seq_len), dtype=bool))
print("因果掩码：")
print(causal_mask.astype(int))

output_causal, weights_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print("
应用掩码后的注意力权重：")
print(np.round(weights_causal, 3))

## 5. 多头注意力（Multi-Head Attention）

单头注意力只能关注一种模式。多头注意力将 {model}$ 分成 $ 个头，
每个头学习不同的关注模式，最后拼接：

83268\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1,...,\text{head}_h)W^O83268

In [ ]:
def multi_head_attention(X, W_q, W_k, W_v, W_o, num_heads):
    seq_len, d_model = X.shape
    d_k = d_model // num_heads
    Q = X @ W_q
    K = X @ W_k
    V = X @ W_v
    Q = Q.reshape(seq_len, num_heads, d_k).transpose(1, 0, 2)
    K = K.reshape(seq_len, num_heads, d_k).transpose(1, 0, 2)
    V = V.reshape(seq_len, num_heads, d_k).transpose(1, 0, 2)
    heads = []
    for i in range(num_heads):
        out, _ = scaled_dot_product_attention(Q[i], K[i], V[i])
        heads.append(out)
    return np.concatenate(heads, axis=-1) @ W_o

d_model, num_heads = 8, 2
X = np.random.randn(seq_len, d_model)
W_q = np.random.randn(d_model, d_model) * 0.1
W_k = np.random.randn(d_model, d_model) * 0.1
W_v = np.random.randn(d_model, d_model) * 0.1
W_o = np.random.randn(d_model, d_model) * 0.1

mha_out = multi_head_attention(X, W_q, W_k, W_v, W_o, num_heads)
print(f"输入形状: {X.shape}")
print(f"多头注意力输出形状: {mha_out.shape}")

## 6. 用 PyTorch 验证

In [ ]:
import torch
import torch.nn as nn

d_model, num_heads, seq_len = 64, 8, 10
mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, batch_first=True)
x = torch.randn(1, seq_len, d_model)
output, attn_weights = mha(x, x, x)
print(f"输入形状:  {x.shape}")
print(f"输出形状:  {output.shape}")
print(f"注意力权重形状: {attn_weights.shape}")

## 7. 小结

| 概念 | 要点 |
|------|------|
| 自注意力 | 每个 token 对所有 token 计算相关度，加权求和 |
| 缩放因子 | 防止点积过大，稳定梯度 |
| 因果掩码 | 解码器生成时只能看过去，不能看未来 |
| 多头注意力 | 并行多个注意力头，捕获不同语义模式 |

下一篇：[02_tokenization.ipynb](02_tokenization.ipynb)